In [1]:
import pandas as pd, os

os.makedirs("../data/powerbi", exist_ok=True)
OUT = "../data/powerbi/"

assets   = pd.read_csv("../data/clean/dim_asset.csv")
crises   = pd.read_csv("../data/clean/dim_crisis.csv")
rolling  = pd.read_parquet("../data/clean/rolling_corr_monthly.parquet")
conn     = pd.read_parquet("../data/clean/connectedness.parquet")
crisis_c = pd.read_parquet("../data/clean/crisis_corr.parquet")
safe     = pd.read_parquet("../data/clean/safe_haven.parquet")
wprices  = pd.read_parquet("../data/clean/prices_weekly.parquet")
wret     = pd.read_parquet("../data/clean/returns_weekly.parquet")
cat = assets.set_index("Asset")["Category"]

def to_long(df, value):
    d = df.copy(); d.index.name = "Date"; d.columns.name = None
    return d.reset_index().melt(id_vars="Date", var_name="Asset", value_name=value).dropna()

def mirror(df):   # fills both halves of the heatmap grid
    m = df.copy()
    m[["Asset_A", "Asset_B"]] = df[["Asset_B", "Asset_A"]].values
    m[["Cat_A", "Cat_B"]] = df[["Cat_B", "Cat_A"]].values
    return pd.concat([df, m], ignore_index=True)

# dimensions
assets[["Asset", "Ticker", "Category"]].to_csv(OUT + "dim_asset.csv", index=False)
crises.to_csv(OUT + "dim_crisis.csv", index=False)

# facts
to_long(wprices, "Close").round(4).to_csv(OUT + "fact_prices_weekly.csv", index=False)
to_long(wret, "Return").round(5).to_csv(OUT + "fact_returns_weekly.csv", index=False)

r = rolling.copy()
r["Cat_A"], r["Cat_B"] = r["Asset_A"].map(cat), r["Asset_B"].map(cat)
r["Correlation"] = r["Correlation"].round(3)
r.to_csv(OUT + "fact_rolling_corr.csv", index=False)

heat = mirror(r.loc[r["Window"] == 120, ["Date", "Asset_A", "Asset_B", "Cat_A", "Cat_B", "Correlation"]])
heat.to_csv(OUT + "fact_corr_matrix.csv", index=False)

mirror(crisis_c).round(3).to_csv(OUT + "fact_crisis_corr.csv", index=False)
conn.round(4).to_csv(OUT + "fact_connectedness.csv", index=False)
safe.round(3).to_csv(OUT + "fact_safe_haven.csv", index=False)

for f in sorted(os.listdir(OUT)):
    print(f"{f:28s} {len(pd.read_csv(OUT + f)):>9,} rows")

C:\Users\User\AppData\Local\Temp\ipykernel_17560\1272219240.py:31: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  to_long(wprices, "Close").round(4).to_csv(OUT + "fact_prices_weekly.csv", index=False)
C:\Users\User\AppData\Local\Temp\ipykernel_17560\1272219240.py:32: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  to_long(wret, "Return").round(5).to_csv(OUT + "fact_returns_weekly.csv", index=False)
C:\Users\User\AppData\Local\Temp\ipykernel_17560\1272219240.py:43: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  conn.round(4).to_csv(OUT + "fact_connectedness.csv", index=False)


dim_asset.csv                       44 rows
dim_crisis.csv                       8 rows
fact_connectedness.csv             318 rows
fact_corr_matrix.csv           494,480 rows
fact_crisis_corr.csv            13,218 rows
fact_prices_weekly.csv          55,530 rows
fact_returns_weekly.csv         55,480 rows
fact_rolling_corr.csv          738,969 rows
fact_safe_haven.csv                 43 rows
